# The Transformation Logic

In [0]:
query = """
SELECT 
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name, 
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ci.gender, 'n/a')
    END AS gender,
    ci.birth_date AS birth_date,
    ci.created_date AS created_date
FROM silver.crm_cust_info AS ci
LEFT JOIN silver.erp_cust_az12 AS ca
    ON ci.customer_number = ca.customer_id
LEFT JOIN silver.erp_loc_a101 AS la
    ON ci.location_id = la.company_id
"""

In [0]:
df = spark.sql(query)
display(df.limit(10))

# Writting Gold Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.gold.dim_customers")
)

# Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10